In [1]:
print("today")

today


In [2]:
!pip install gemmi

   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.3 MB ? eta -:--:--
   ------------------ --------------------- 1.0/2.3 MB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 2.3/2.3 MB 2.9 MB/s  0:00:01


In [12]:
from pathlib import Path
import gemmi



def get_method(cif_path):
    doc = gemmi.cif.read(str(cif_path))
    block = doc.sole_block()
    methods = block.find_values("_exptl.method")
    method = "; ".join(methods) if methods else "UNKNOWN"

    return method

In [13]:
from utils import get_protein_ids

In [ ]:
protein_ids = get_protein_ids(f"dataset\\nrPDB-GO_2019.06.18_{split}_sequences.fasta")


In [ ]:
protein_ids[0]

{'entry_id': '192L', 'full_id': '192L-A', 'chain': 'A'}

In [ ]:
from collections import defaultdict
from tqdm.auto import tqdm

def count_methods_per_split(pdb_split):
    cached_method_info = {}
    skipped = set()
    errors = {}
    accepted_values = {"train", "val", "test"}
    if pdb_split not in accepted_values:
        raise ValueError("pdb_split should be one of: train, val, or test")
    
    methods_freq_count = defaultdict(int)

    protein_id_path = Path(f"dataset/nrPDB-GO_2019.06.18_{pdb_split}_sequences.fasta")
    protein_ids = get_protein_ids(protein_id_path)

    for protein in tqdm(protein_ids, desc = "Methods counted"):
        

        cif_path = Path(f"structures/pdb/pdb_{pdb_split}/{protein['entry_id']}.cif")
        entry_id = protein["entry_id"]

        if cif_path.exists():
            if entry_id not in cached_method_info:
                try: 
                    cached_method_info[entry_id] = get_method(cif_path)
                except Exception as err:
                    errors[entry_id] = f"There was an error with {entry_id}: {err}"
                    continue
                method = cached_method_info[entry_id]
                methods_freq_count[method] += 1
                continue
            else: 
                method = cached_method_info[entry_id]
                methods_freq_count[method] += 1
            
        else:
            skipped.add(entry_id)

        
           
    return methods_freq_count, skipped, errors
    


In [ ]:
def delete_non_xray_structures(pdb_split): 
    accepted_values = ["train", "val", "test"]
    if pdb_split not in accepted_values:
        raise ValueError("pdb_split should be one of: train, val, or test")

    protein_id_path = Path(f"dataset/nrPDB-GO_2019.06.18_{pdb_split}_sequences.fasta")
    protein_ids = get_protein_ids(protein_id_path)
    unique_ids = {protein["entry_id"] for protein in protein_ids}

    deleted = set()
    failed = {}
    cif_files_not_found = set()
    for unique_id in tqdm(unique_ids, desc = f"Deleting non-xray structures from pdb_{pdb_split}"):

        cif_path = Path(f"structures/pdb/pdb_{pdb_split}/{unique_id}.cif")
    

        if not cif_path.exists():
            cif_files_not_found.add(unique_id)
            continue
            
        try:
            method = get_method(cif_path)
        except Exception as err:
            failed[unique_id] = f"There was an error parsing the cif file {err}"
            continue
        
        try:
            if "X-RAY DIFFRACTION" not in method:
                cif_path.unlink()
                deleted.add(unique_id)
        except Exception as err:
            failed[unique_id] = f"There was an error deleting the file {err}"
            continue
        
            
    print(f"{len(deleted)} files were successfully deleted")
    print(f"{len(cif_files_not_found)} cif files weren't found in the pdb {pdb_split} directory")
    print(f"{len(failed)} cif files couldn't be processed")

    return deleted, failed, cif_files_not_found

In [ ]:
count_methods_per_split("train")

Just reached file 100. 29802 files left. Still going...
Just reached file 200. 29702 files left. Still going...
Just reached file 300. 29602 files left. Still going...
Just reached file 400. 29502 files left. Still going...
Just reached file 500. 29402 files left. Still going...
Just reached file 600. 29302 files left. Still going...
Just reached file 700. 29202 files left. Still going...
Just reached file 800. 29102 files left. Still going...
Just reached file 900. 29002 files left. Still going...
Just reached file 1000. 28902 files left. Still going...
Just reached file 1100. 28802 files left. Still going...
Just reached file 1200. 28702 files left. Still going...
Just reached file 1300. 28602 files left. Still going...
Just reached file 1400. 28502 files left. Still going...
Just reached file 1500. 28402 files left. Still going...
Just reached file 1600. 28302 files left. Still going...
Just reached file 1700. 28202 files left. Still going...
Just reached file 1800. 28102 files left

{"'X-RAY DIFFRACTION'": 24588,
 "'SOLUTION NMR'": 2268,
 "'FIBER DIFFRACTION'": 4,
 "'ELECTRON MICROSCOPY'": 3003,
 "'SOLUTION NMR'; 'THEORETICAL MODEL'": 4,
 "'ELECTRON CRYSTALLOGRAPHY'": 10,
 "'SOLUTION SCATTERING'": 4,
 "'SOLUTION NMR'; 'SOLUTION SCATTERING'": 4,
 "'SOLID-STATE NMR'": 5,
 "'SOLID-STATE NMR'; 'ELECTRON MICROSCOPY'": 2,
 "'ELECTRON MICROSCOPY'; 'SOLUTION SCATTERING'": 1,
 "'X-RAY DIFFRACTION'; EPR": 1,
 "'X-RAY DIFFRACTION'; 'NEUTRON DIFFRACTION'": 5,
 "'NEUTRON DIFFRACTION'; 'X-RAY DIFFRACTION'": 1,
 "'NEUTRON DIFFRACTION'": 1,
 "'X-RAY DIFFRACTION'; 'SOLUTION SCATTERING'": 1}

In [ ]:
count_methods_per_split("val")

Just reached file 100. 3222 files left. Still going...
Just reached file 200. 3122 files left. Still going...
Just reached file 300. 3022 files left. Still going...
Just reached file 400. 2922 files left. Still going...
Just reached file 500. 2822 files left. Still going...
Just reached file 600. 2722 files left. Still going...
Just reached file 700. 2622 files left. Still going...
Just reached file 800. 2522 files left. Still going...
Just reached file 900. 2422 files left. Still going...
Just reached file 1000. 2322 files left. Still going...
Just reached file 1100. 2222 files left. Still going...
Just reached file 1200. 2122 files left. Still going...
Just reached file 1300. 2022 files left. Still going...
Just reached file 1400. 1922 files left. Still going...
Just reached file 1500. 1822 files left. Still going...
Just reached file 1600. 1722 files left. Still going...
Just reached file 1700. 1622 files left. Still going...
Just reached file 1800. 1522 files left. Still going...
J

{"'X-RAY DIFFRACTION'": 2756,
 "'SOLUTION NMR'": 232,
 "'ELECTRON MICROSCOPY'": 329,
 "'SOLID-STATE NMR'": 2,
 "'ELECTRON CRYSTALLOGRAPHY'": 1,
 "'X-RAY DIFFRACTION'; 'NEUTRON DIFFRACTION'": 1,
 "'SOLUTION NMR'; 'SOLUTION SCATTERING'": 1}

In [14]:
count_methods_per_split("test")

Methods counted:   0%|          | 0/3416 [00:00<?, ?it/s]

(defaultdict(int,
             {"'X-RAY DIFFRACTION'": 2552,
              "'ELECTRON MICROSCOPY'": 705,
              "'SOLUTION NMR'": 146,
              "'ELECTRON CRYSTALLOGRAPHY'": 6,
              "'SOLUTION SCATTERING'": 5,
              "'ELECTRON MICROSCOPY'; 'SOLUTION SCATTERING'": 1,
              "'SOLID-STATE NMR'": 1}),
 set(),
 {})

In [15]:
methods = {"'X-RAY DIFFRACTION'": 24588,
 "'SOLUTION NMR'": 2268,
 "'FIBER DIFFRACTION'": 4,
 "'ELECTRON MICROSCOPY'": 3003,
 "'SOLUTION NMR'; 'THEORETICAL MODEL'": 4,
 "'ELECTRON CRYSTALLOGRAPHY'": 10,
 "'SOLUTION SCATTERING'": 4,
 "'SOLUTION NMR'; 'SOLUTION SCATTERING'": 4,
 "'SOLID-STATE NMR'": 5,
 "'SOLID-STATE NMR'; 'ELECTRON MICROSCOPY'": 2,
 "'ELECTRON MICROSCOPY'; 'SOLUTION SCATTERING'": 1,
 "'X-RAY DIFFRACTION'; EPR": 1,
 "'X-RAY DIFFRACTION'; 'NEUTRON DIFFRACTION'": 5,
 "'NEUTRON DIFFRACTION'; 'X-RAY DIFFRACTION'": 1,
 "'NEUTRON DIFFRACTION'": 1,
 "'X-RAY DIFFRACTION'; 'SOLUTION SCATTERING'": 1}

In [23]:
for key, _ in methods.items():
    if "X-RAY DIFFRACTION" in key:
        print(key)

'X-RAY DIFFRACTION'
'X-RAY DIFFRACTION'; EPR
'X-RAY DIFFRACTION'; 'NEUTRON DIFFRACTION'
'NEUTRON DIFFRACTION'; 'X-RAY DIFFRACTION'
'X-RAY DIFFRACTION'; 'SOLUTION SCATTERING'
